# GEIM for MSFR: external sensors

The sensors are placed externally using the Generalised Empirical Interpolation Method

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

data_root = os.environ.get("NUSHRED_DATA_DIR", "../../NuSHRED_Datasets")
path_snaps = os.path.join(data_root, "MSFR/")

import sys
sys.path.append('../P2/')

idx_params, params, var_names, is_vector, fom_times, rescaling_snaps, Nmodes = pickle.load(open(path_snaps+'msfr_p.uloff', 'rb'))
fom_times = np.array(fom_times)

Let us import the mesh

In [ ]:
import pyvista as pv
from IPython.display import clear_output

domain_grid = pv.read(os.path.join(path_snaps, "evol_domain.vtk"))
mesh = domain_grid.points

measured_field = 1
assert not is_vector[measured_field], "GEIM requires a scalar field."

clear_output()
print('The observable field is '+var_names[measured_field])

# Reflector region: outside the core rectangle x in [0, 2.05], y in [-1.13, 1.13]
# (matches the geometry drawn by plot_msfr_regions in Code/P2/plots.py)
refl_tags = np.where((mesh[:, 0] > 2.05) | (np.abs(mesh[:, 2]) > 1.13))[0]
Nh = mesh.shape[0]

Let us import the snapshots

In [ ]:
from pyforce.tools.functions_list import FunctionsList, train_test_split

field = var_names[measured_field]
path_compressed = os.path.join(path_snaps, "CompressedDataset")

svd_basis = pickle.load(open(os.path.join(path_compressed, f"pod_basis_{field}.svd"), "rb"))
svd_sing_vals = pickle.load(open(os.path.join(path_compressed, f"sing_vals_{field}.svd"), "rb"))
svd_coeff = pickle.load(open(os.path.join(path_compressed, "v_POD_all_fields.svd"), "rb"))

# Reconstruct the snapshots for a single parameter (tau) from the SVD/POD decomposition,
# undoing the rescaling applied before compression (same procedure as Code/P2/01_svd.ipynb)
param_idx = 0
Nt = svd_coeff[field].shape[1]

rec = np.zeros((Nh, Nt))
for tt in range(Nt):
    rec[:, tt] = rescaling_snaps[measured_field].inverse_transform(
        np.linalg.multi_dot([svd_basis, svd_sing_vals, svd_coeff[field][param_idx, tt].T]).reshape(-1, 1)
    ).flatten()

snaps = FunctionsList(dofs=Nh)
for tt in range(Nt):
    snaps.append(rec[:, tt])

times = fom_times

train_snaps = FunctionsList(dofs=Nh)
test_snaps  = FunctionsList(dofs=Nh)

train_time = list()
test_time  = list()

for ii in range(len(snaps)):
    if (ii+1) % 2:
        train_snaps.append(snaps(ii))
        train_time.append(times[ii])
    else:
        test_snaps.append(snaps(ii))
        test_time.append(times[ii])

assert len(train_snaps) + len(test_snaps) == len(snaps)
assert len(train_time) + len(test_time) == len(times)
assert len(train_snaps) == len(train_time)
assert len(test_snaps) == len(test_time)

## Offline Phase
The EIM is used to generate a reduced space and to find the optimal positions for sensors

In [ ]:
from pyforce.offline.geim import GEIM

point_spread = 0.025

geim_class = GEIM(domain_grid, gdim=3, varname=field, sensors_type='Gaussian')

Mmax = 15
_xm_list = [mesh[tag] for tag in refl_tags]
max_abs_err, max_rel_err, beta_coeff = geim_class.fit(train_snaps, Mmax, sensor_params={'s': point_spread, 'xm_list': _xm_list}, verbose=True)

Let us plot the locations of sensors

In [ ]:
from plots import plot_msfr_regions

fig, axs = plt.subplots(figsize=(6, 6))

plot_msfr_regions(axs, show_ticks=True)

# Sensor locations: peak of each magic sensor's Gaussian kernel
sensor_nodes = geim_class.magic_sensors.nodes
mag_idx = np.argmax(geim_class.magic_sensors.library.return_matrix(), axis=0)
mag_pts = sensor_nodes[mag_idx]

axs.scatter(mag_pts[:,0], mag_pts[:,2], color='r')
for i in range(mag_pts.shape[0]):
    axs.text(mag_pts[i, 0], mag_pts[i, 2], str(i+1), c='white', fontsize=12, ha='left')

axs.set_xticks([])
axs.set_yticks([])

Let us create some measurements from the training data

In [ ]:
from scipy.interpolate import CubicSpline

num_sensors = 3
snap_measurements = np.zeros((num_sensors, len(train_snaps), 1))

for tt in range(len(train_snaps)):
    snap_measurements[:, tt, 0] = geim_class.magic_sensors.action(train_snaps(tt))[:num_sensors, 0]

Let us plot the measurement

In [ ]:
t = train_time
newnt = 2000
new_t = np.linspace(t[0], t[-1], newnt)

noise_value = 1/100
Xsensor = np.zeros((num_sensors, newnt, 1))
Xsensor2 = np.zeros((num_sensors, newnt, 1))

for j in range(num_sensors):
    
    measure = snap_measurements[j, :, 0]
    
    spline = CubicSpline(t, measure)
    Xsensor[j, :, 0] = spline(new_t) * (1 + np.random.normal(scale=noise_value, size=new_t.shape))
    Xsensor2[j, :, 0] = spline(new_t) + np.random.normal(scale=noise_value / 7, size=new_t.shape)
    
fig, axs = plt.subplots(ncols = 2, figsize=(10,4))
axs[0].plot(new_t, Xsensor[:,:,0].T, label='Spline')
axs[0].plot(train_time, snap_measurements[:,:,0].T, '--', label='FOM')

axs[1].plot(new_t, Xsensor2[:,:,0].T, label='Spline')
axs[1].plot(train_time, snap_measurements[:,:,0].T, '--', label='FOM')

Let us store the data

In [ ]:
path_svd = './OfflineSVD/'
os.makedirs(path_svd, exist_ok=True)

measurements = dict()
measurements['mesh'] = mesh
measurements['location'] = mag_pts[:num_sensors]
measurements['point_spread'] = point_spread
measurements['noise'] = noise_value

measurements['measured_field'] = measured_field

measurements['noisy_output'] = Xsensor

pickle.dump(measurements, open(path_svd+'measurements_GEIM.data', 'wb'))

## Online Phase
Let us test the reconstruction of the EIM algorithm for unseen data

In [ ]:
from pyforce.online.geim import GEIM as GEIMOnline

geim_online = GEIMOnline(domain_grid, gdim=3, varname=field)
geim_online.set_basis(basis=geim_class.magic_functions)
geim_online.set_magic_sensors(sensors=geim_class.magic_sensors.library)
geim_online.set_tikhonov_matrices(beta_coeffs=beta_coeff)

Mmax_test = 10
noise_value = 5/7/100
reg_params = {'type': 'tikhonov', 'lambda': noise_value**2}

results = geim_online.compute_errors(test_snaps, Mmax=Mmax_test, noise_std=noise_value, regularization_params=reg_params, verbose=True)
ave_test_abs_err, ave_test_rel_err = results.mean_abs_err, results.mean_rel_err

pickle.dump([ave_test_abs_err.mean(), ave_test_rel_err.mean()], open(path_svd+'test_errors.geim', 'wb'))

Let us plot the average errors

In [ ]:
plt.semilogy(ave_test_rel_err)
print(ave_test_rel_err.mean())

Let us make a contour plot

In [ ]:
from plots import plot_contour
import matplotlib.gridspec as gridspec

idx = 100

measures = geim_online.get_measurements(test_snaps(idx), M=Mmax_test, noise_std=noise_value)
rec = geim_online.estimate(measures, regularization_params=reg_params)(0)
res = test_snaps(idx) - rec

levels = np.linspace(test_snaps(idx).min()*0.975, test_snaps(idx).max()*1.025, 40)

# Create a grid specification with 2 rows and 3 columns, leaving space for the colorbar
fig = plt.figure(figsize=(12, 5))
gs = gridspec.GridSpec(2, 3, height_ratios=[4, 0.1])  # Last row for colorbar

# Plot FOM and TR-GEIM, sharing the same colorbar
ax0 = plt.subplot(gs[0, 0])
fom_ = plot_contour(ax0, mesh[:, [0,2]], test_snaps(idx), cmap=cm.plasma, levels=levels)
ax0.set_title('FOM')

ax1 = plt.subplot(gs[0, 1])
geim_ = plot_contour(ax1, mesh[:, [0,2]], rec, cmap=cm.plasma, levels=levels)
ax1.set_title('TR-GEIM')

# Plot Residual with its own colorbar
ax2 = plt.subplot(gs[0, 2])
residual_ = plot_contour(ax2, mesh[:, [0,2]], res, cmap=cm.hot)
ax2.set_title('Residual')
plt.colorbar(residual_, ax=ax2)

# Create a single horizontal colorbar for FOM and TR-GEIM, placed below the first two subplots
cbar_ax = plt.subplot(gs[1, 0:2])  # Span the first two columns
cbar = fig.colorbar(fom_, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Colorbar for FOM and TR-GEIM')